In [1]:
from dotenv import load_dotenv

load_dotenv()
import math
import os

import wikipedia as wiki
from langchain.agents import create_agent
from langchain_community.tools import DuckDuckGoSearchResults, WikipediaQueryRun
from langchain_community.utilities import (
    DuckDuckGoSearchAPIWrapper,
    WikipediaAPIWrapper,
)
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

/var/folders/q0/_ryr9r1174qdpbwpfztmyx6c0000gp/T/ipykernel_28762/2102146147.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchResults


In [ ]:
llm = ChatOpenAI(
    model=os.getenv("GPT_MODEL", "your-gpt-model"),
    base_url=os.getenv("PORTKEY_BASE_URL", "your-base-url"),
    api_key=os.getenv("PORTKEY_API_KEY", "your-api-key"),
)

In [3]:
# Define Calculator Tool
@tool
def calculator(expression):
    """
    A simple calculator that can evaluate mathematical expressions.
    Use this for any math calculations.
    """
    try:
        # Clean the expression
        expression = expression.replace("^", "**")  # Handle exponentiation
        # Safe evaluation of mathematical expressions
        allowed_names = {
            k: v for k, v in math.__dict__.items() if not k.startswith("__")
        }
        allowed_names.update({"abs": abs, "round": round, "min": min, "max": max})

        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"The result is: {result}"
    except Exception as e:
        return f"Error in calculation: {e!s}"

In [4]:
# Initialize search tools

search = DuckDuckGoSearchResults(api_wrapper=DuckDuckGoSearchAPIWrapper())
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
wiki.set_user_agent("agents-hands-on/0.1 (s.chamezopoulos@elsevier.com)")

In [5]:
search.run("Ford Motor Company")

"snippet: History of the Ford Motor Company Henry Ford (pictured c. 1919), founded and led the company, presiding over it during two tenures, 1906-1919 and 1943-1945 The Ford Motor Company is an American automaker, the world's fifth largest based on worldwide vehicle sales., title: History of the Ford Motor Company - Wikipedia, link: https://en.wikipedia.org/wiki/History_of_the_Ford_Motor_Company, snippet: Henry Ford, founder of Ford Motor Company Ford Motor Company is an American multinational automobile manufacturer founded by Henry Ford and incorporated in Michigan on June 16,..., title: Ford Motor Company, link: https://grokipedia.com/page/Ford_Motor_Company, snippet: Ford Racing Is Older Than Ford Motor Company. This New Podcast Revisits Its Most Important Chapters 3 MIN READ More Stories Oct 9, 2026, title: @FordOnline - at.ford.com, link: https://www.at.ford.com/us/en/home.html, snippet: Get the latest Ford news and official press releases straight from the source. Explore major

In [6]:
wikipedia.run("Ford Motor Company")

'Page: Ford Motor Company\nSummary: The Ford Motor Company (commonly known as Ford, sometimes abbreviated as FoMoCo) is an American multinational automobile manufacturer headquartered in Dearborn, Michigan, United States. It was founded by Henry Ford and incorporated on June 16, 1903. The company sells automobiles and commercial vehicles under the Ford brand, and luxury cars under its Lincoln brand. The company is listed on the New York Stock Exchange under the single-letter ticker symbol F and is controlled by the Ford family. They have minority ownership but a plurality of the voting power.\nFord introduced methods for large-scale manufacturing of cars and large-scale management of an industrial workforce using elaborately engineered manufacturing sequences typified by moving assembly lines. By 1914, these methods were known around the world as Fordism. Ford\'s former British subsidiaries Jaguar and Land Rover, acquired in 1989 and 2000, respectively, were sold to the Indian automake

In [7]:
# Create tools list
tools = [calculator, search, wikipedia]

In [8]:
# Initialize the agent
agent = create_agent(
    llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Use the tools for math, current information, and background facts.",
    checkpointer=InMemorySaver(),  # this is the "buffer memory"
)

In [9]:
# Demo function to run examples
def ask(agent, q: str) -> str:
    return agent.invoke({"messages": [("human", q)]})["messages"][-1].content


def run_demo():
    """Run demonstration examples"""

    print(" Multi-Tool Agent Demo")
    print("=" * 50)

    examples = [
        "What is 15 * 23 + 45?",
        "Who is the current president of France and what is the square root of 144?",
        "Search for the latest news about Agentic AI developments in 2025",
        "Tell me about Albert Einstein and calculate 299792458 / 1000000",
        "What is the weather like today in Hyderabad, India and what is 2^10?",
    ]

    for i, question in enumerate(examples, 1):
        print(f"\n Example {i}: {question}")
        print("-" * 40)
        try:
            response = ask(agent, question)
            print(f" Response: {response}")
        except Exception as e:
            print(f" Error: {e!s}")
        print("\n" + "=" * 50)

In [10]:
# Interactive function
def chat_with_agent():
    """Interactive chat with the agent"""
    print("\n Interactive Chat Mode")
    print("Type 'quit' to exit")
    print("-" * 30)

    while True:
        user_input = input("\n You: ").strip()

        if user_input.lower() in ["quit", "exit", "bye"]:
            print(" Goodbye!")
            break

        if user_input:
            try:
                response = ask(agent, user_input)
                print(f" Agent: {response}")
            except Exception as e:
                print(f" Error: {e!s}")

In [11]:
# Main execution
if __name__ == "__main__":
    print(" Setting up Multi-Tool Agent...")

    # Run demo
    choice = input("\nRun demo examples? (y/n): ").lower().strip()
    if choice == "y":
        run_demo()

    # Start interactive mode
    choice = input("\nStart interactive chat? (y/n): ").lower().strip()
    if choice == "y":
        chat_with_agent()

 Setting up Multi-Tool Agent...
 Multi-Tool Agent Demo

 Example 1: What is 15 * 23 + 45?
----------------------------------------
 Error: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id


 Example 2: Who is the current president of France and what is the square root of 144?
----------------------------------------
 Error: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id


 Example 3: Search for the latest news about Agentic AI developments in 2025
----------------------------------------
 Error: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id


 Example 4: Tell me about Albert Einstein and calculate 299792458 / 1000000
----------------------------------------
 Error: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id


 Example 5: What i